In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import os 
import re

df = pd.read_csv("../data/training_data/mixed_demographics.csv")
df.shape




(900, 9)

In [ ]:
df.head()

,index,MI,age,BMI,height,weight,diastolic_BP,systolic_BP,sex
0,0,healthy,62.593012,25.312186,161.647749,64.949107,76.057670,137.765801,False
1,1,healthy,58.037959,32.100795,172.543391,94.715477,84.272113,143.459426,False
2,2,pMI,65.840496,26.099055,166.521430,71.650562,75.340419,137.100866,False
3,3,pMI,68.518300,27.672836,166.176884,75.779414,75.697258,140.518093,True
4,4,healthy,55.591381,27.266429,161.314821,70.185424,79.973837,133.740780,False


In [ ]:
df["MI"].value_counts()

MI
healthy    452
pMI        448
Name: count, dtype: int64

In [ ]:
df.isna().sum()

index           0
MI              0
age             0
BMI             0
height          0
weight          0
diastolic_BP    0
systolic_BP     0
sex             0
dtype: int64

In [ ]:


bad_folder = "../data/training_data/impossible_samples"

bad_files = [
    f for f in os.listdir(bad_folder)
    if f.endswith(".npy")
]

bad_indices = [
    int(re.match(r"\d+", f).group())
    for f in bad_files
]

print("Number of bad subjects:", len(bad_indices))
print(sorted(bad_indices))

Number of bad subjects: 38
[57, 75, 113, 138, 139, 172, 174, 289, 292, 308, 326, 335, 378, 385, 400, 424, 426, 439, 445, 503, 508, 573, 591, 601, 658, 666, 673, 693, 699, 722, 754, 776, 793, 814, 822, 841, 846, 875]


In [ ]:


df_clean = df[~df["index"].isin(bad_indices)].copy()

print("Original dataset:", df.shape)
print("Clean dataset:", df_clean.shape)

print(
    "Bad subjects remaining:",
    df_clean["index"].isin(bad_indices).sum()
)

Original dataset: (900, 9)
Clean dataset: (862, 9)
Bad subjects remaining: 0


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

def run_pca(folder, files, components=32):
    samples = [
        np.load(os.path.join(folder, filename), mmap_mode="r")
        for filename in files
    ]

    encoder = OneHotEncoder(sparse_output=False)

    region_arrays = [
        sample[:, :, 3].reshape(-1, 1)
        for sample in samples
    ]

    encoder.fit(np.vstack(region_arrays))

    X = np.array([
        np.concatenate([
            sample[:, :, :3].ravel(),
            encoder.transform(region).ravel()
        ])
        for sample, region in zip(samples, region_arrays)
    ])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=components)
    X_pca = pca.fit_transform(X_scaled)

    return X_pca, pca, scaler, encoder

In [ ]:
folder = "../data/training_data/clean_samples"
files = [f for f in os.listdir(folder) if f.endswith(".npy")]
files = sorted(files, key=lambda f: int(re.match(r"\d+", f).group()))

X_pca, pca, scaler, encoder = run_pca(folder, files, components=32)
print("PCA shape:", X_pca.shape)

In [ ]:
pc_features = [f"PC{i}" for i in range(1, 33)]

features = [
    "age",
    "BMI",
    "height",
    "weight",
    "diastolic_BP",
    "systolic_BP",
    "sex"
] + pc_features

print(len(features), "features:", features)

df_clean = df_clean.drop(columns=pc_features, errors="ignore")


X = df_clean[features]
y = df_clean['MI'].map({
    "healthy":0,
    "pMI":1
})

print(X.shape)

X.head()


39 features: ['age', 'BMI', 'height', 'weight', 'diastolic_BP', 'systolic_BP', 'sex', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20', 'PC21', 'PC22', 'PC23', 'PC24', 'PC25', 'PC26', 'PC27', 'PC28', 'PC29', 'PC30', 'PC31', 'PC32']


In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

X shape: (862, 71)
y shape: (862,)
MI
0    436
1    426
Name: count, dtype: int64


In [ ]:
y.head()

0    0
1    0
2    1
3    1
4    0
Name: MI, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size = 150, 
    random_state = 9888, 
    stratify = y
)

print("Development set:", X_dev.shape)
print("Test set:", X_test.shape)

print("\n Development classes:")
print(y_dev.value_counts())

print("\nTest classes:")
print(y_test.value_counts())

Development set: (712, 71)
Test set: (150, 71)

 Development classes:
MI
0    360
1    352
Name: count, dtype: int64

Test classes:
MI
0    76
1    74
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits = 5, 
    shuffle = True, 
    random_state = 9888
)

for fold, (train_idx, val_idx) in enumerate(cv.split(X_dev, y_dev), start=1):
    print(
        f"Fold {fold}: "
        f"train = {len(train_idx)}, "
        f"validation = {len(val_idx)}"
    )

Fold 1: train = 569, validation = 143
Fold 2: train = 569, validation = 143
Fold 3: train = 570, validation = 142
Fold 4: train = 570, validation = 142
Fold 5: train = 570, validation = 142


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [ ]:
from sklearn.model_selection import cross_val_score

accuracy_scores = cross_val_score(
    logreg,
    X_dev,
    y_dev,
    cv=cv,
    scoring="accuracy"
)

print(accuracy_scores)
print("Mean accuracy:", accuracy_scores.mean())

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/lsabourian/Intro_to_ML_PCyay/.venv/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 860, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lsabourian/Intro_to_ML_PCyay/.venv/lib/python3.14/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Users/lsabourian/Intro_to_ML_PCyay/.venv/lib/python3.14/site-packages/sklearn/pipeline.py", line 632, in fit
    Xt = self._fit(
        X, y, routed_params, raw_params=params, callback_ctx=callback_ctx
    )
  File "/Users

In [ ]:
from sklearn.model_selection import cross_validate

scoring = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

scores = cross_validate(
    logreg,
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    values = scores[f"test_{metric}"]
    print(f"{metric}: {values.mean():.3f} ± {values.std():.3f}")

accuracy: 0.791 ± 0.008
precision: 0.784 ± 0.006
recall: 0.795 ± 0.018
f1: 0.790 ± 0.010
roc_auc: 0.859 ± 0.023
